# Train the full model set on Colab — 5 component specialists + 6 condition specialists

**Architecture (12 models): `pole` (reused) + 5 component + 6 condition.**

| group | subset | classes | model |
|---|---|---|---|
| component | comp_wire | wire | yolo26x |
| component | comp_insulator | h_insulator, v_insulator | yolo26x |
| component | comp_crossarm | crossarm_stright, top_crossarm, om_crossarm | yolo26x |
| component | comp_vegetation | vegetation | yolo26m |
| component | comp_rust | rust | yolo26m |
| condition | cond_v_insulator | normal/band/broken/chip_off | yolo26m |
| condition | cond_h_insulator | normal/broken/chip_off | yolo26m |
| condition | cond_straight_crossarm | normal/band | yolo26m |
| condition | cond_top_crossarm | normal/band | yolo26m |
| condition | cond_om_crossarm | normal/band | yolo26m |
| condition | cond_wire | wire_normal/cross_wire | yolo26m |

Data-rich detectors (wire/insulator/crossarm) -> **yolo26x**; small/data-limited (vegetation, rust,
all condition families) -> **yolo26m** (resists overfit). `pole` is NOT retrained.

**Upload:** zip each subset folder and drop the 11 `.zip`s into your Drive folder **`MyDrive/finalGo/`** (e.g. `comp_insulator.zip`, `cond_v_insulator.zip`, ...). The loader auto-handles either zip layout (folder-at-root **or** contents-at-root) and strips the macOS `__MACOSX/` + `._*` artifacts that otherwise crash YOLO. **Runtime -> A100 -> Run all.**

In [ ]:
# @title 1) Reduce CUDA fragmentation (before torch) + GPU check
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "no nvidia-smi")
assert torch.cuda.is_available(), "No CUDA GPU! Runtime -> Change runtime type -> A100."
NAME=torch.cuda.get_device_name(0); VRAM=torch.cuda.get_device_properties(0).total_memory/1e9
print(f"GPU: {NAME} | VRAM {VRAM:.0f} GB | torch {torch.__version__} CUDA {torch.version.cuda}")

In [ ]:
# @title 2) Repo + deps
REPO="/content/repo"
!git clone https://github.com/arupa444/trainDronisight.git $REPO 2>/dev/null || (cd $REPO && git pull)
%cd $REPO
!pip -q install uv && uv pip install --system -e .
import torch; print("CUDA:", torch.cuda.is_available())

In [ ]:
# @title 3) Mount Drive
from google.colab import drive; drive.mount('/content/drive')

In [ ]:
# @title 4) Load the 11 trainable subsets from Drive (per-subset zips) -> /content/data
import os, glob, zipfile, shutil
DRIVE_DATA = "/content/drive/MyDrive/finalGo"   # <-- Drive folder holding the per-subset .zip files
SUBSETS = ["comp_wire","comp_insulator","comp_crossarm","comp_vegetation","comp_rust",
           "cond_v_insulator","cond_h_insulator","cond_straight_crossarm","cond_top_crossarm",
           "cond_om_crossarm","cond_wire"]
DEST  = "/content/data/yolo_train_db"; os.makedirs(DEST, exist_ok=True)
STAGE = "/content/_stage"

def _strip_mac(root):
    # macOS zips carry __MACOSX/ + ._* AppleDouble + .DS_Store; YOLO crashes reading ._*.jpg -> purge
    for p in glob.glob(f"{root}/**/__MACOSX", recursive=True): shutil.rmtree(p, ignore_errors=True)
    for pat in ("._*", ".DS_Store"):
        for p in glob.glob(f"{root}/**/{pat}", recursive=True):
            try: os.remove(p)
            except OSError: pass

def _find_db_dir(root):
    # the real subset dir = shallowest dir holding BOTH images/ and labels/ (ignores __MACOSX)
    cands = [d for d, subs, _ in os.walk(root)
             if "__MACOSX" not in d and {"images","labels"} <= set(subs)]
    cands.sort(key=lambda p: p.count(os.sep))
    return cands[0] if cands else None

def locate(sub):
    for c in (f"{DRIVE_DATA}/{sub}.zip", f"{DRIVE_DATA}/yolo_train_db/{sub}.zip",
              f"{DRIVE_DATA}/{sub}", f"{DRIVE_DATA}/yolo_train_db/{sub}"):
        if os.path.exists(c): return c
    h = sorted(glob.glob(f"{DRIVE_DATA}/**/{sub}.zip", recursive=True))
    return h[0] if h else None

missing = []
for sub in SUBSETS:
    dst = f"{DEST}/{sub}"
    if os.path.isdir(dst) and glob.glob(f"{dst}/images/train/**/*.jpg", recursive=True):
        print("present:", sub); continue
    src = locate(sub)
    if not src: missing.append(sub); print("MISSING :", sub); continue
    st = f"{STAGE}/{sub}"; shutil.rmtree(st, ignore_errors=True); os.makedirs(st, exist_ok=True)
    if src.endswith(".zip"):
        with zipfile.ZipFile(src) as z: z.extractall(st)
    else:
        shutil.copytree(src, st, dirs_exist_ok=True)
    _strip_mac(st)
    dbdir = _find_db_dir(st)
    assert dbdir, f"{sub}: no images/+labels/ inside {src}"
    shutil.rmtree(dst, ignore_errors=True); shutil.move(dbdir, dst); _strip_mac(dst)
    shutil.rmtree(st, ignore_errors=True)
    print("loaded  :", sub, "->", dst)
assert not missing, f"Upload these as <name>.zip into {DRIVE_DATA}: {missing}"

os.environ["DRONISIGHT_DATA"] = "/content/data"
import importlib, shared.config as C; importlib.reload(C)
for sub in SUBSETS:
    for c in glob.glob(f"{DEST}/{sub}/**/*.cache", recursive=True): os.remove(c)  # poisoned-cache invariant
    tr = len(glob.glob(f"{DEST}/{sub}/images/train/clahe/*.jpg"))
    va = len(glob.glob(f"{DEST}/{sub}/images/val/clahe/*.jpg"))
    print(f"{sub:24s} train(clahe)={tr:5d}  val(clahe)={va:4d}")


In [ ]:
# @title 5) Train all 11 (data-rich -> yolo26x, small -> yolo26m). Saves to Drive after EACH.
from train_yolo.train_components import run
from notebooks.colab_utils import save_runs_to_drive
DRIVE_DATA = globals().get("DRIVE_DATA", "/content/drive/MyDrive/finalGo")  # runs/ go here too
PLAN = [   # (subset, model, imgsz, epochs)
    ("comp_wire",              "yolo26x.pt", 1280, 150),
    ("comp_insulator",         "yolo26x.pt", 1280, 150),
    ("comp_crossarm",          "yolo26x.pt", 1280, 150),
    ("comp_vegetation",        "yolo26m.pt", 1280, 150),
    ("comp_rust",              "yolo26m.pt", 1280, 200),   # data-limited -> a few more epochs
    ("cond_v_insulator",       "yolo26m.pt", 1280, 150),
    ("cond_h_insulator",       "yolo26m.pt", 1280, 150),
    ("cond_straight_crossarm", "yolo26m.pt", 1280, 150),
    ("cond_top_crossarm",      "yolo26m.pt", 1280, 150),
    ("cond_om_crossarm",       "yolo26m.pt", 1280, 150),
    ("cond_wire",              "yolo26m.pt", 1280, 150),
]
def pick_batch(model, vram):
    heavy = "26x" in model
    table = [(78,6),(40,3),(22,2),(0,1)] if heavy else [(78,32),(40,16),(22,8),(0,4)]
    for thr,b in table:
        if vram>=thr: return b
    return 1
for sub, model, imgsz, epochs in PLAN:
    batch = pick_batch(model, VRAM)
    print("="*70, f"\nTRAIN {sub}  model={model} imgsz={imgsz} batch={batch} epochs={epochs}\n", "="*70)
    run(subset=sub, version="clahe", epochs=epochs, imgsz=imgsz, batch=batch, model=model)
    print("saved to Drive:", save_runs_to_drive(drive_root=DRIVE_DATA))


In [ ]:
# @title 6) Per-model val mAP
import glob, os
from train_yolo.eval_yolo import evaluate
for sub in ["comp_wire","comp_insulator","comp_crossarm","comp_vegetation","comp_rust",
            "cond_v_insulator","cond_h_insulator","cond_straight_crossarm","cond_top_crossarm",
            "cond_om_crossarm","cond_wire"]:
    w=sorted(glob.glob(f"runs/**/{sub}/**/weights/best.pt", recursive=True), key=os.path.getmtime)
    if w: print(evaluate(w[-1], sub, split="val", imgsz=1280), "\n")